In [1]:
import pandas as pd

ratings = pd.read_csv(
    r"C:\surya\ml\Movie-Recommendation-System\data\raw\ratings.dat",
    sep="::",
    engine="python",
    names=["userId","movieId","rating","timestamp"]
)

movies = pd.read_csv(
    r"C:\surya\ml\Movie-Recommendation-System\data\raw/movies.dat",
    sep="::",
    engine="python",
    encoding="latin-1",
    names=["movieId","title","genres"]
)

In [2]:
user_movie_matrix = ratings.pivot_table(
    index="userId",
    columns="movieId",
    values="rating"
)

user_movie_matrix.head()

movieId,1,2,3,4,5,6,7,8,9,10,...,3943,3944,3945,3946,3947,3948,3949,3950,3951,3952
userId,,,,,,,,,,,,,,,,,,,,,
1,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,2.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
user_movie_matrix.shape

(6040, 3706)

In [4]:
user_movie_matrix_filled = user_movie_matrix.fillna(0)


In [5]:
from sklearn.metrics.pairwise import cosine_similarity

user_similarity = cosine_similarity(
    user_movie_matrix_filled
)

In [6]:
user_similarity.shape

(6040, 6040)

In [7]:
user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_movie_matrix_filled.index,
    columns=user_movie_matrix_filled.index
)

In [8]:
user_similarity_df.loc[1].sort_values(
    ascending=False
).head(10)

userId
1       1.000000
5343    0.412117
5190    0.411899
1481    0.392110
1283    0.386597
5705    0.360898
5762    0.355600
1359    0.350329
1476    0.339496
541     0.334864
Name: 1, dtype: float64

In [9]:
similar_users = user_similarity_df.loc[1].sort_values(
    ascending=False
)

similar_users.head(10)

userId
1       1.000000
5343    0.412117
5190    0.411899
1481    0.392110
1283    0.386597
5705    0.360898
5762    0.355600
1359    0.350329
1476    0.339496
541     0.334864
Name: 1, dtype: float64

In [10]:
user_1_movies = ratings[
    ratings["userId"] == 1
]["movieId"]

print("Movies rated by User 1:")
print(len(user_1_movies))

Movies rated by User 1:
53


In [11]:
similar_user_movies = ratings[
    ratings["userId"] == 5343
]

similar_user_movies.head()

,userId,movieId,rating,timestamp
884674,5343,588,5,966890638
884675,5343,1,5,966890616
884676,5343,593,5,960682228
884677,5343,594,5,966890616
884678,5343,595,5,966890616


In [12]:
top_movies = similar_user_movies[
    similar_user_movies["rating"] >= 4
]
print("Movies rated 4 or higher by User 5343:")
print(len(top_movies))  
print(top_movies[["movieId", "rating"]].head(10))

Movies rated 4 or higher by User 5343:
34
        movieId  rating
884674      588       5
884675        1       5
884676      593       5
884677      594       5
884678      595       5
884679     2078       5
884680     2080       5
884681     2081       5
884682     2096       4
884683      910       5


In [13]:
recommendations = top_movies[
    ~top_movies["movieId"].isin(user_1_movies)
]

In [14]:
recommendations = recommendations.merge(
    movies,
    on="movieId"
)

recommendations[
    ["title", "rating"]
].head(20)

,title,rating
0,"Silence of the Lambs, The (1991)",5
1,"Jungle Book, The (1967)",5
2,Lady and the Tramp (1955),5
3,"Little Mermaid, The (1989)",5
4,Sleeping Beauty (1959),4
5,Some Like It Hot (1959),5
6,Animal House (1978),4
7,American Beauty (1999),5
8,Charlotte's Web (1973),4
9,Gladiator (2000),5


In [15]:
movie_user_matrix = ratings.pivot_table(
    index="movieId",
    columns="userId",
    values="rating"
)

movie_user_matrix = movie_user_matrix.fillna(0)

movie_user_matrix.shape

(3706, 6040)

In [16]:
from sklearn.metrics.pairwise import cosine_similarity

movie_similarity = cosine_similarity(
    movie_user_matrix
)

movie_similarity.shape

(3706, 3706)

In [17]:
movie_similarity_df = pd.DataFrame(
    movie_similarity,
    index=movie_user_matrix.index,
    columns=movie_user_matrix.index
)

In [18]:
movie_name = "Toy Story (1995)"

In [19]:
movie_id = movies[
    movies["title"] == movie_name
]["movieId"].iloc[0]

print(movie_id)

1


In [20]:
movie_similarity_df[movie_id]\
    .sort_values(ascending=False)\
    .head(10)

movieId
1       1.000000
3114    0.633104
1265    0.610826
588     0.605849
2355    0.579382
1270    0.570125
34      0.563637
1196    0.552856
1580    0.552362
356     0.551034
Name: 1, dtype: float64

In [21]:
print(movie_user_matrix.shape)
print(movie_similarity.shape)

(3706, 6040)
(3706, 3706)


In [22]:
movie_indices = pd.Series(
    movies["movieId"].values,
    index=movies["title"]
)

In [23]:
def get_similar_movies(movie_title, n=10):

    movie_id = movie_indices[movie_title]

    similarity_scores = movie_similarity_df[
        movie_id
    ]

    similarity_scores = similarity_scores.sort_values(
        ascending=False
    )

    similar_movie_ids = similarity_scores.index[
        1:n+1
    ]

    recommendations = movies[
        movies["movieId"].isin(
            similar_movie_ids
        )
    ]

    return recommendations[
        ["title", "genres"]
    ]

In [24]:
get_similar_movies(
    "Toy Story (1995)"
)

,title,genres
33,Babe (1995),Children's|Comedy|Drama
352,Forrest Gump (1994),Comedy|Romance|War
584,Aladdin (1992),Animation|Children's|Comedy|Musical
1178,Star Wars: Episode V - The Empire Strikes Back...,Action|Adventure|Drama|Sci-Fi|War
1245,Groundhog Day (1993),Comedy|Romance
1250,Back to the Future (1985),Comedy|Sci-Fi
1539,Men in Black (1997),Action|Adventure|Comedy|Sci-Fi
2286,"Bug's Life, A (1998)",Animation|Children's|Comedy
2502,"Matrix, The (1999)",Action|Sci-Fi|Thriller
3045,Toy Story 2 (1999),Animation|Children's|Comedy
